# 06 — Revision Checks

Methodological robustness requested by supervisor. All outputs → `results/tables/revision/`. Do **not** modify notebooks 02–05.

In [1]:
import subprocess, sys, importlib

def _ensure(pkg, import_as=None):
    name = import_as or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

_ensure("doubleml")

import warnings, pathlib, itertools
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMRegressor, LGBMClassifier
from scipy.stats import norm as _norm
from doubleml import DoubleMLIRM, DoubleMLData
import doubleml, econml, lightgbm, sklearn

print("doubleml :", doubleml.__version__)
print("econml   :", econml.__version__)
print("lightgbm :", lightgbm.__version__)
print("sklearn  :", sklearn.__version__)

FIGURES = pathlib.Path("../results/figures")
TABLES  = pathlib.Path("../results/tables")
REVDIR  = TABLES / "revision"
DATA    = pathlib.Path("../data/processed")
REVDIR.mkdir(parents=True, exist_ok=True)


doubleml : 0.11.4
econml   : 0.16.0
lightgbm : 4.6.0
sklearn  : 1.6.1


In [2]:
# ── Shared constants (identical to 03/05) ─────────────────────────────────────
LGBM_PARAMS = dict(n_estimators=400, learning_rate=0.05, num_leaves=63,
                   colsample_bytree=0.8, subsample=0.8, verbose=-1)
ORDINAL  = ["OrgSize", "Age", "EdLevel"]
NUMERIC  = ["WorkExp", "YearsCode", "CompLog"]
MISS_IND = ["WorkExp_missing", "YearsCode_missing", "CompLog_missing"]
NOMINAL  = ["DevType", "Country", "Industry", "RemoteWork", "ICorPM"]
EXP3_BINS   = [0, 2, 20, 100];  EXP3_LABELS = ["0-2 yrs", "3-20 yrs", "21+ yrs"]
EXP4_BINS   = [0, 2, 5, 20, 100]; EXP4_LABELS = ["0-2 yrs", "3-5 yrs", "6-20 yrs", "21+ yrs"]

def make_kf():
    return KFold(n_splits=5, shuffle=True, random_state=42)

# ── Data loading ───────────────────────────────────────────────────────────────
def load_year(parquet_path):
    df = pd.read_parquet(parquet_path)
    if "adoption_status" in df.columns:
        df = df[df["adoption_status"].isin(
            ["adopter","non_adopter","plan_to_soon"])].copy()
        df["treated"] = df["treated"].fillna(0.0)
        df = df.reset_index(drop=True)
    cv = df["Country"].value_counts()
    df["Country"] = df["Country"].where(~df["Country"].isin(cv[cv<50].index), "Other")
    X_nom = pd.get_dummies(df[NOMINAL], drop_first=False, dtype=float)
    X_num = df[ORDINAL + NUMERIC + MISS_IND].astype(float)
    X = pd.concat([X_num, X_nom], axis=1).reset_index(drop=True)
    Y = df["JobSat"].values.astype(float)
    T = df["treated"].values.astype(float)
    df_narrow = None
    if "adoption_status" in df.columns:
        df_narrow = df[df["adoption_status"] != "plan_to_soon"].reset_index(drop=True).copy()
    return df, X, Y, T, df_narrow

def load_year_prespec(parquet_path):
    "Same as load_year but X uses only pre-determined covariates."
    df, X_full, Y, T, df_narrow = load_year(parquet_path)
    PRE = ["Age","EdLevel","WorkExp","YearsCode","WorkExp_missing","YearsCode_missing"]
    X_num_pre = df[PRE].astype(float)
    X_nom_pre = pd.get_dummies(df[["Country"]], drop_first=False, dtype=float)
    cv_pre = df["Country"].value_counts()   # already collapsed by load_year
    X_pre = pd.concat([X_num_pre, X_nom_pre], axis=1).reset_index(drop=True)
    return df, X_pre, Y, T

# ── LR diagnostic PS (same as cell 9 of 03/05) ───────────────────────────────
def lr_ps(X, T):
    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X.values)
    lr     = LogisticRegression(max_iter=1000, C=0.1, random_state=42, n_jobs=-1)
    lr.fit(X_sc, T)
    ps = lr.predict_proba(X_sc)[:, 1]
    return ps

# ── DML cross-fitting ─────────────────────────────────────────────────────────
def cross_fit(X, Y, T, kf=None):
    if kf is None: kf = make_kf()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        Y_hat = cross_val_predict(
            LGBMRegressor(**LGBM_PARAMS, random_state=42), X.values, Y, cv=kf)
        T_hat = cross_val_predict(
            LGBMClassifier(**LGBM_PARAMS, random_state=42), X.values, T, cv=kf,
            method="predict_proba")[:, 1]
    return Y_hat, T_hat, Y - Y_hat, T - T_hat

# ── PLR ATE ───────────────────────────────────────────────────────────────────
def plr_ate(Y_res, T_res):
    ols = sm.OLS(Y_res, sm.add_constant(pd.DataFrame({"T_res": T_res}))
                 ).fit(cov_type="HC3")
    return dict(ate=float(ols.params["T_res"]), se=float(ols.bse["T_res"]),
                ci_lo=float(ols.conf_int().loc["T_res",0]),
                ci_hi=float(ols.conf_int().loc["T_res",1]),
                p=float(ols.pvalues["T_res"]))

# ── AIPW score helpers ────────────────────────────────────────────────────────
def _aipw_compute(Y, T, T_hat, mu1, mu0, clip, winsorise):
    e = np.clip(T_hat, clip, 1-clip)
    scores = mu1 - mu0 + T*(Y-mu1)/e - (1-T)*(Y-mu0)/(1-e)
    if winsorise:
        lo, hi = np.percentile(scores, [2.5, 97.5])
        scores = np.clip(scores, lo, hi)
    return scores

def aipw_slearner(X, Y, T, T_hat, kf, clip, winsorise):
    Xa = X.values
    mu1, mu0 = np.zeros(len(Y)), np.zeros(len(Y))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for tr, te in kf.split(Xa):
            reg = LGBMRegressor(**LGBM_PARAMS, random_state=42)
            reg.fit(np.column_stack([Xa[tr], T[tr]]), Y[tr])
            mu1[te] = reg.predict(np.column_stack([Xa[te], np.ones(len(te))]))
            mu0[te] = reg.predict(np.column_stack([Xa[te], np.zeros(len(te))]))
    return _aipw_compute(Y, T, T_hat, mu1, mu0, clip, winsorise)

def aipw_tlearner(X, Y, T, T_hat, kf, clip, winsorise):
    Xa = X.values
    mu1, mu0 = np.zeros(len(Y)), np.zeros(len(Y))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for tr, te in kf.split(Xa):
            m1 = T[tr]==1; m0 = T[tr]==0
            r1 = LGBMRegressor(**LGBM_PARAMS, random_state=42)
            r1.fit(Xa[tr][m1], Y[tr][m1])
            r0 = LGBMRegressor(**LGBM_PARAMS, random_state=42)
            r0.fit(Xa[tr][m0], Y[tr][m0])
            mu1[te] = r1.predict(Xa[te])
            mu0[te] = r0.predict(Xa[te])
    return _aipw_compute(Y, T, T_hat, mu1, mu0, clip, winsorise)

def summarise_scores(scores):
    n = len(scores)
    ate = float(scores.mean())
    se  = float(scores.std(ddof=1) / n**0.5)
    return dict(ate=ate, se=se, ci_lo=ate-1.96*se, ci_hi=ate+1.96*se)

# ── DoubleML IRM ──────────────────────────────────────────────────────────────
def run_irm(X, Y, T, kf_smpls, score="ATE", trim=0.01):
    dml_data = DoubleMLData.from_arrays(X.values, Y, T)
    irm = DoubleMLIRM(dml_data,
                      ml_g=LGBMRegressor(**LGBM_PARAMS, random_state=42),
                      ml_m=LGBMClassifier(**LGBM_PARAMS, random_state=42),
                      score=score, trimming_rule="truncate",
                      trimming_threshold=trim, draw_sample_splitting=False)
    irm.set_sample_splitting(kf_smpls)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        irm.fit()
    theta = float(irm.coef[0])
    se    = float(irm.se[0])
    ci    = irm.confint()
    return dict(ate=theta, se=se,
                ci_lo=float(ci.iloc[0,0]), ci_hi=float(ci.iloc[0,1]),
                irm_obj=irm)

# ── Band effects (residual approach) ─────────────────────────────────────────
def band_effects_residual(Y_res, T_res, bands_series, labels):
    "Y_res ~ T_res * C(band), HC3. Returns effects, joint Wald chi2, contrasts."
    df_m = pd.DataFrame({
        "Y_res": Y_res, "T_res": T_res,
        "band": pd.Categorical(bands_series.values, categories=labels)
    }).dropna()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mod = smf.ols("Y_res ~ T_res * C(band)", data=df_m).fit(cov_type="HC3")

    plist = list(mod.params.index)
    cov   = mod.cov_params()

    def gate_and_se(lbl):
        g = float(mod.params["T_res"])
        v = cov.loc["T_res","T_res"]
        int_nm = f"T_res:C(band)[T.{lbl}]"
        if int_nm in plist:
            g += float(mod.params[int_nm])
            v += cov.loc[int_nm,int_nm] + 2*cov.loc["T_res",int_nm]
        return g, v**0.5

    effects = {}
    for lbl in labels:
        g, s = gate_and_se(lbl)
        z = g/s; p = 2*(1-_norm.cdf(abs(z)))
        effects[lbl] = dict(gate=g, se=s,
                            ci_lo=g-1.96*s, ci_hi=g+1.96*s, p=p,
                            n=int((df_m["band"]==lbl).sum()))

    int_names = [n for n in plist if "T_res:C(band)" in n]
    R_j = np.zeros((len(int_names), len(plist)))
    for i, nm in enumerate(int_names):
        R_j[i, plist.index(nm)] = 1.0
    wj  = mod.wald_test(R_j, scalar=True)
    joint = dict(chi2=float(wj.statistic), df=len(int_names), p=float(wj.pvalue))

    def contrast(lbl_a, lbl_b):
        R = np.zeros(len(plist))
        int_a = f"T_res:C(band)[T.{lbl_a}]"
        int_b = f"T_res:C(band)[T.{lbl_b}]"
        if lbl_a != labels[0] and int_a in plist: R[plist.index(int_a)] = 1.0
        if lbl_b != labels[0] and int_b in plist: R[plist.index(int_b)] = -1.0
        t  = mod.t_test(R)
        ci = t.conf_int()
        return dict(diff=float(t.effect[0]), se=float(t.sd[0]),
                    ci_lo=float(ci[0,0]), ci_hi=float(ci[0,1]),
                    p=float(np.atleast_1d(t.pvalue).ravel()[0]))

    contrasts = {}
    if len(labels) == 3:
        contrasts["mid_vs_junior"] = contrast(labels[1], labels[0])
        contrasts["mid_vs_senior"] = contrast(labels[1], labels[2])
    else:
        contrasts["6-20_vs_0-2"]  = contrast(labels[2], labels[0])
        contrasts["6-20_vs_21+"]  = contrast(labels[2], labels[3])

    return effects, joint, contrasts

# ── Band effects (DR-score approach) ─────────────────────────────────────────
def band_effects_dr(scores, bands_series, labels):
    "Regress DR scores on band dummies (no intercept), HC3."
    df_m = pd.DataFrame({"sc": scores,
                         "band": pd.Categorical(bands_series.values, categories=labels)
                        }).dropna()
    dummies = pd.get_dummies(df_m["band"], prefix="b", drop_first=False, dtype=float)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mod = sm.OLS(df_m["sc"], dummies).fit(cov_type="HC3")

    effects = {}
    for lbl in labels:
        col = f"b_{lbl}"
        if col not in mod.params: continue
        g = float(mod.params[col]); s = float(mod.bse[col])
        z = g/s; p = 2*(1-_norm.cdf(abs(z)))
        effects[lbl] = dict(gate=g, se=s, ci_lo=g-1.96*s, ci_hi=g+1.96*s, p=p,
                            n=int((df_m["band"]==lbl).sum()))

    plist = list(mod.params.index)
    cov   = mod.cov_params()

    def _contrast_dr(lbl_a, lbl_b):
        ca = f"b_{lbl_a}"; cb = f"b_{lbl_b}"
        if ca not in plist or cb not in plist: return None
        diff = float(mod.params[ca] - mod.params[cb])
        se   = (cov.loc[ca,ca] + cov.loc[cb,cb] - 2*cov.loc[ca,cb])**0.5
        z = diff/se; p = 2*(1-_norm.cdf(abs(z)))
        return dict(diff=diff, se=float(se), ci_lo=diff-1.96*se, ci_hi=diff+1.96*se, p=p)

    contrasts = {}
    if len(labels) == 3:
        contrasts["mid_vs_junior"] = _contrast_dr(labels[1], labels[0])
        contrasts["mid_vs_senior"] = _contrast_dr(labels[1], labels[2])
    else:
        contrasts["6-20_vs_0-2"]  = _contrast_dr(labels[2], labels[0])
        contrasts["6-20_vs_21+"]  = _contrast_dr(labels[2], labels[3])

    plist2 = list(mod.params.index)
    cols   = [f"b_{lbl}" for lbl in labels if f"b_{lbl}" in plist2]
    grand  = float(mod.params[cols].mean())
    R_j    = np.zeros((len(cols)-1, len(plist2)))
    for i in range(len(cols)-1):
        R_j[i, plist2.index(cols[i])]   =  1.0
        R_j[i, plist2.index(cols[i+1])] = -1.0
    wj = mod.wald_test(R_j, scalar=True)
    joint = dict(chi2=float(wj.statistic), df=len(cols)-1, p=float(wj.pvalue),
                 mean_score=grand)

    return effects, joint, contrasts

# ── Formatting helpers ─────────────────────────────────────────────────────────
def _sig(p):
    return "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns"

def _fmt_row(label, d):
    return (f"  {label:50s}  {d['ate']:+.3f}  [{d['ci_lo']:+.3f},{d['ci_hi']:+.3f}]"
            f"  SE={d['se']:.3f}")

def _fmt_band(label, e):
    return (f"  {label:12s}  n={e['n']:>6,}  GATE={e['gate']:+.3f}"
            f"  [{e['ci_lo']:+.3f},{e['ci_hi']:+.3f}]  {_sig(e['p'])}")

def _fmt_contrast(name, c):
    if c is None: return f"  {name}: —"
    return (f"  {name:20s}  Δ={c['diff']:+.3f}  [{c['ci_lo']:+.3f},{c['ci_hi']:+.3f}]"
            f"  p={c['p']:.3f}  {_sig(c['p'])}")

print("Helper functions loaded.")


Helper functions loaded.


## Section 0 — Reproduce PLR ATE

In [3]:
print("=" * 60)
print("Section 0 — Reproduce PLR ATE (assert before continuing)")
print("=" * 60)

YEARS = {
    2025: DATA / "analysis.parquet",
    2026: DATA / "analysis_2026.parquet",
}
EXPECTED_ATE = {2025: 0.143, 2026: 0.212}  # 3-decimal check

year_data = {}  # cache for all sections

for yr, path in YEARS.items():
    df, X, Y, T, df_narrow = load_year(path)
    kf = make_kf()
    Y_hat, T_hat, Y_res, T_res = cross_fit(X, Y, T, kf=kf)
    result = plr_ate(Y_res, T_res)
    kf_smpls = list(kf.split(X))

    # Store for downstream sections
    year_data[yr] = dict(df=df, X=X, Y=Y, T=T, df_narrow=df_narrow,
                         Y_hat=Y_hat, T_hat=T_hat, Y_res=Y_res, T_res=T_res,
                         kf=kf, kf_smpls=kf_smpls)

    rounded = round(result["ate"], 3)
    exp     = EXPECTED_ATE[yr]
    ok      = "✓" if rounded == exp else f"✗  expected {exp}"
    print(f"{yr}: ATE={result['ate']:.4f}  SE={result['se']:.4f}"
          f"  [{result['ci_lo']:+.4f},{result['ci_hi']:+.4f}]  {ok}")
    assert rounded == exp, f"{yr} ATE mismatch: got {rounded}, expected {exp}"

print("\nAssertions passed.")


Section 0 — Reproduce PLR ATE (assert before continuing)


2025: ATE=0.1433  SE=0.0349  [+0.0749,+0.2117]  ✓


2026: ATE=0.2115  SE=0.0634  [+0.0873,+0.3358]  ✓

Assertions passed.


## Section A — DML vs AIPW Decomposition

13-row table per year: PLR, AIPW grid (2 outcome models × 2 clips × 2 winsorise), DoubleMLIRM ×3, overlap-weighted, causal forest.

In [4]:
print("=" * 70)
print("Section A — AIPW / IRM decomposition")
print("=" * 70)

CATE_PATHS = {
    2025: (DATA / "cates.npy",        DATA / "cf_model.joblib"),
    2026: (DATA / "cates_2026.npy",   DATA / "cf_model_2026.joblib"),
}
N_EXPECTED = {2025: 20373, 2026: 11262}

sec_a_rows   = {}  # year -> list of dicts
sec_a_irm_dr = {}  # year -> DR scores for Section B
sec_a_cates  = {}  # year -> cates array

for yr in [2025, 2026]:
    d = year_data[yr]
    X, Y, T, T_hat = d["X"], d["Y"], d["T"], d["T_hat"]
    Y_res, T_res, kf, kf_smpls = d["Y_res"], d["T_res"], d["kf"], d["kf_smpls"]
    n = len(Y)

    rows = []

    # 1. PLR
    plr = plr_ate(Y_res, T_res)
    rows.append({"label": "1. PLR (main)", **plr})

    # 2-9. AIPW grid: S/T-learner × clip 0.05/0.01 × winsorise on/off
    grid = list(itertools.product(
        [("S-learner", aipw_slearner), ("T-learner", aipw_tlearner)],
        [0.05, 0.01],
        [True, False]
    ))
    row_num = 2
    irm_dr_scores = None
    for (oname, ofunc), clip, wins in grid:
        label = f"{row_num}. AIPW {oname} clip={clip} wins={'yes' if wins else 'no'}"
        sc = ofunc(X, Y, T, T_hat, kf, clip, wins)
        s  = summarise_scores(sc)
        rows.append({"label": label, **s})
        # Keep T-learner, clip=0.01, no-winsorise for Section B DR approach
        if oname == "T-learner" and clip == 0.01 and not wins:
            irm_dr_scores = sc
        row_num += 1

    sec_a_irm_dr[yr] = irm_dr_scores

    # 10. DoubleMLIRM ATE trim=0.01
    r10 = run_irm(X, Y, T, kf_smpls, score="ATE", trim=0.01)
    rows.append({"label": "10. DoubleMLIRM ATE trim=0.01", **{k:v for k,v in r10.items() if k!="irm_obj"}})
    irm_obj_01 = r10["irm_obj"]

    # 11. DoubleMLIRM ATE trim=0.05
    r11 = run_irm(X, Y, T, kf_smpls, score="ATE", trim=0.05)
    rows.append({"label": "11. DoubleMLIRM ATE trim=0.05", **{k:v for k,v in r11.items() if k!="irm_obj"}})

    # 12. DoubleMLIRM ATTE
    r12 = run_irm(X, Y, T, kf_smpls, score="ATTE", trim=0.01)
    rows.append({"label": "12. DoubleMLIRM ATTE trim=0.01", **{k:v for k,v in r12.items() if k!="irm_obj"}})

    # 13. Overlap-weighted mean (T-learner, no-wins, clip=0.01)
    ow_scores = sec_a_irm_dr[yr]
    ow_wts    = T_hat * (1 - T_hat)
    ow_wts    = ow_wts / ow_wts.sum()
    ow_ate    = float((ow_scores * ow_wts * n).sum() / 1.0)   # rescale: sum(w)=1 so mean = sum(sc*w*n)/n
    # Actually overlap weighted ATE = sum(scores * e*(1-e)) / sum(e*(1-e))
    ow_wts2   = T_hat * (1 - T_hat)
    ow_ate2   = float(np.average(ow_scores, weights=ow_wts2))
    se_ow     = float(np.sqrt(np.average((ow_scores - ow_ate2)**2, weights=ow_wts2) / n))
    rows.append({"label": "13. OW-AIPW e(1-e)-weighted", "ate": ow_ate2, "se": se_ow,
                 "ci_lo": ow_ate2-1.96*se_ow, "ci_hi": ow_ate2+1.96*se_ow})

    # 14. Causal forest: mean CATE + OW mean CATE
    cates_path, _ = CATE_PATHS[yr]
    cates = np.load(cates_path)
    if len(cates) == n:
        cf_ate   = float(cates.mean())
        cf_se    = float(cates.std(ddof=1) / n**0.5)
        cf_ow    = float(np.average(cates, weights=T_hat*(1-T_hat)))
        cf_ow_se = float(np.sqrt(np.average((cates-cf_ow)**2, weights=T_hat*(1-T_hat)) / n))
        rows.append({"label": "14. CF mean CATE (cached)", "ate": cf_ate, "se": cf_se,
                     "ci_lo": cf_ate-1.96*cf_se, "ci_hi": cf_ate+1.96*cf_se})
        rows.append({"label": "14b. CF OW mean CATE", "ate": cf_ow, "se": cf_ow_se,
                     "ci_lo": cf_ow-1.96*cf_ow_se, "ci_hi": cf_ow+1.96*cf_ow_se})
        sec_a_cates[yr] = cates
    else:
        print(f"  WARNING: cates.npy length {len(cates)} != n={n}; CF rows skipped")

    # Extreme PS diagnostics
    lgbm_ext = ((T_hat < 0.05) | (T_hat > 0.95)).mean()
    ps_diag  = lr_ps(X, T)
    lr_ext   = ((ps_diag < 0.05) | (ps_diag > 0.95)).mean()
    cate_hi  = float(cates[T_hat > 0.9].mean()) if yr in sec_a_cates else float("nan")
    cate_lo  = float(cates[T_hat <= 0.9].mean()) if yr in sec_a_cates else float("nan")

    sec_a_rows[yr] = rows

    # Print
    print(f"\n{'─'*70}")
    print(f"  {yr}  (n={n:,})  LGBM extreme PS: {lgbm_ext:.1%}  LR extreme: {lr_ext:.1%}")
    print(f"  CATE mean (T_hat>0.9): {cate_hi:+.3f}  CATE mean (rest): {cate_lo:+.3f}")
    print(f"{'─'*70}")
    print(f"  {'Estimator':50s}  {'ATE':>7s}  {'95% CI':>18s}  SE")
    print(f"  {'─'*50}  {'─'*7}  {'─'*18}  {'─'*5}")
    for r in rows:
        p = r.get("p", float("nan"))
        sig = f"  {_sig(p)}" if not np.isnan(p) else ""
        print(f"  {r['label']:50s}  {r['ate']:+.3f}  [{r['ci_lo']:+.3f},{r['ci_hi']:+.3f}]"
              f"  {r['se']:.3f}{sig}")

    # Verdict
    plr_v   = rows[0]["ate"]
    aipw_s_v = rows[1]["ate"]   # S-learner, clip=0.05, wins=yes (current)
    tl_v    = [r for r in rows if "T-learner" in r["label"] and "0.01" in r["label"] and "no" in r["label"]][0]["ate"]
    print(f"\n  Verdict {yr}: PLR={plr_v:+.3f}  AIPW-S(0.05,wns)={aipw_s_v:+.3f}"
          f"  AIPW-T(0.01,no)={tl_v:+.3f}")
    s_no_wins = rows[2]["ate"]   # S-learner, clip=0.05, wins=no
    t_no_wins = rows[6]["ate"]   # T-learner, clip=0.05, wins=no
    if abs(aipw_s_v - plr_v) > 0.05:
        print(f"  → Winsorising is the primary driver: without winsorising, "
              f"S-learner ({s_no_wins:+.3f}) ≈ T-learner ({t_no_wins:+.3f}) ≈ PLR ({plr_v:+.3f}). "
              f"Choice of outcome model (S vs T) is secondary.")

    # Save CSV
    df_out = pd.DataFrame(rows)[["label","ate","se","ci_lo","ci_hi"]]
    df_out.to_csv(REVDIR / f"aipw_decomposition_{yr}.csv", index=False)
    print(f"  Saved → aipw_decomposition_{yr}.csv")


Section A — AIPW / IRM decomposition


/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:119: DeprecationWarning: 'trimming_threshold' is deprecated and will be removed in a future version. Use 'ps_processor_config' with 'clipping_threshold' instead.
  irm = DoubleMLIRM(dml_data,



──────────────────────────────────────────────────────────────────────
  2025  (n=20,373)  LGBM extreme PS: 11.9%  LR extreme: 2.8%
  CATE mean (T_hat>0.9): +0.094  CATE mean (rest): +0.136
──────────────────────────────────────────────────────────────────────
  Estimator                                               ATE              95% CI  SE
  ──────────────────────────────────────────────────  ───────  ──────────────────  ─────
  1. PLR (main)                                       +0.143  [+0.075,+0.212]  0.035  ***
  2. AIPW S-learner clip=0.05 wins=yes                +0.011  [-0.044,+0.067]  0.028
  3. AIPW S-learner clip=0.05 wins=no                 +0.163  [+0.059,+0.266]  0.053
  4. AIPW S-learner clip=0.01 wins=yes                +0.015  [-0.041,+0.072]  0.029
  5. AIPW S-learner clip=0.01 wins=no                 +0.227  [+0.073,+0.380]  0.078
  6. AIPW T-learner clip=0.05 wins=yes                +0.034  [-0.024,+0.092]  0.030
  7. AIPW T-learner clip=0.05 wins=no           

/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:119: DeprecationWarning: 'trimming_threshold' is deprecated and will be removed in a future version. Use 'ps_processor_config' with 'clipping_threshold' instead.
  irm = DoubleMLIRM(dml_data,



──────────────────────────────────────────────────────────────────────
  2026  (n=11,262)  LGBM extreme PS: 50.0%  LR extreme: 22.5%
  CATE mean (T_hat>0.9): +0.207  CATE mean (rest): +0.153
──────────────────────────────────────────────────────────────────────
  Estimator                                               ATE              95% CI  SE
  ──────────────────────────────────────────────────  ───────  ──────────────────  ─────
  1. PLR (main)                                       +0.212  [+0.087,+0.336]  0.063  ***
  2. AIPW S-learner clip=0.05 wins=yes                +0.013  [-0.072,+0.098]  0.043
  3. AIPW S-learner clip=0.05 wins=no                 +0.345  [+0.140,+0.550]  0.104
  4. AIPW S-learner clip=0.01 wins=yes                +0.029  [-0.065,+0.123]  0.048
  5. AIPW S-learner clip=0.01 wins=no                 +0.534  [-0.036,+1.105]  0.291
  6. AIPW T-learner clip=0.05 wins=yes                +0.081  [-0.009,+0.171]  0.046
  7. AIPW T-learner clip=0.05 wins=no          

## Section B — Formal Tests of Experience Heterogeneity

(a) Residual approach  (b) DR-score approach

In [5]:
print("=" * 70)
print("Section B — Experience heterogeneity (3-band and 4-band, both years)")
print("=" * 70)

sec_b_summary = {}  # for final summary cell

for yr in [2025, 2026]:
    d = year_data[yr]
    df, Y_res, T_res = d["df"], d["Y_res"], d["T_res"]
    dr_scores = sec_a_irm_dr[yr]

    we = df["WorkExp"].values
    b3 = pd.cut(pd.Series(we), bins=EXP3_BINS, labels=EXP3_LABELS,
                right=True, include_lowest=True)
    b4 = pd.cut(pd.Series(we), bins=EXP4_BINS, labels=EXP4_LABELS,
                right=True, include_lowest=True)

    sec_b_summary[yr] = {}

    for bands_label, bands_series, labels in [
        ("3-band", b3, EXP3_LABELS),
        ("4-band", b4, EXP4_LABELS),
    ]:
        print(f"\n{'─'*70}")
        print(f"  {yr} · {bands_label}")
        print(f"{'─'*70}")

        # (a) Residual approach
        effects_r, joint_r, contrasts_r = band_effects_residual(
            Y_res, T_res, bands_series, labels)
        print(f"  (a) Residual  Wald χ²({joint_r['df']})={joint_r['chi2']:.3f}  p={joint_r['p']:.3f}  {_sig(joint_r['p'])}")
        for lbl in labels:
            print(_fmt_band(lbl, effects_r[lbl]))
        for name, c in contrasts_r.items():
            print(_fmt_contrast(name, c))

        # (b) DR-score approach
        effects_d, joint_d, contrasts_d = band_effects_dr(
            dr_scores, bands_series, labels)
        print(f"\n  (b) DR-score  Wald χ²({joint_d['df']})={joint_d['chi2']:.3f}  p={joint_d['p']:.3f}  {_sig(joint_d['p'])}")
        print(f"      mean DR score = {joint_d['mean_score']:+.3f}  "
              f"(IRM row-10 theta = {sec_a_rows[yr][9]['ate']:+.3f})")
        for lbl in labels:
            if lbl in effects_d:
                print(_fmt_band(lbl, effects_d[lbl]))
        for name, c in contrasts_d.items():
            print(_fmt_contrast(name, c))

        sec_b_summary[yr][bands_label] = dict(
            residual=dict(effects=effects_r, joint=joint_r, contrasts=contrasts_r),
            dr=dict(effects=effects_d, joint=joint_d, contrasts=contrasts_d)
        )

        # Save CSV
        rows_out = []
        for lbl in labels:
            er = effects_r.get(lbl, {}); ed = effects_d.get(lbl, {})
            rows_out.append({
                "band": lbl,
                "n": er.get("n",""),
                "gate_residual": er.get("gate",""), "se_residual": er.get("se",""),
                "gate_dr": ed.get("gate",""), "se_dr": ed.get("se",""),
            })
        fname = f"band_effects_{bands_label.replace('-','')}_b{yr}.csv"
        pd.DataFrame(rows_out).to_csv(REVDIR / fname, index=False)

print("\nSection B done.")


Section B — Experience heterogeneity (3-band and 4-band, both years)

──────────────────────────────────────────────────────────────────────
  2025 · 3-band
──────────────────────────────────────────────────────────────────────
  (a) Residual  Wald χ²(2)=5.090  p=0.078  ns
  0-2 yrs       n= 1,539  GATE=-0.111  [-0.415,+0.193]  ns
  3-20 yrs      n=14,558  GATE=+0.191  [+0.107,+0.274]  ***
  21+ yrs       n= 4,276  GATE=+0.073  [-0.056,+0.201]  ns
  mid_vs_junior         Δ=+0.302  [-0.013,+0.617]  p=0.060  ns
  mid_vs_senior         Δ=+0.118  [-0.035,+0.272]  p=0.131  ns

  (b) DR-score  Wald χ²(2)=0.261  p=0.878  ns
      mean DR score = +0.274  (IRM row-10 theta = +0.221)
  0-2 yrs       n= 1,539  GATE=+0.433  [-0.531,+1.396]  ns
  3-20 yrs      n=14,558  GATE=+0.211  [+0.006,+0.415]  *
  21+ yrs       n= 4,276  GATE=+0.179  [-0.068,+0.425]  ns
  mid_vs_junior         Δ=-0.222  [-1.207,+0.763]  p=0.659  ns
  mid_vs_senior         Δ=+0.032  [-0.289,+0.352]  p=0.846  ns

──────────────

/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),
/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),
/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operatio

## Section C — Common Support Restrictions

Three trimming rules; re-cross-fit nuisances on trimmed sample.

In [6]:
print("=" * 70)
print("Section C — Common support restrictions")
print("=" * 70)

TRIM_RULES = [
    ("LR [0.05,0.95]",   "lr",   0.05, 0.95),
    ("LR [0.10,0.90]",   "lr",   0.10, 0.90),
    ("LGBM [0.05,0.95]", "lgbm", 0.05, 0.95),
]

sec_c_summary = {}

for yr in [2025, 2026]:
    d  = year_data[yr]
    df, X, Y, T = d["df"], d["X"], d["Y"], d["T"]
    ps_lr   = lr_ps(X, T)
    T_hat   = d["T_hat"]

    sec_c_summary[yr] = {}
    we = df["WorkExp"].values
    b4 = pd.cut(pd.Series(we), bins=EXP4_BINS, labels=EXP4_LABELS,
                right=True, include_lowest=True)

    rows_out = []
    for rule_name, ps_type, lo, hi in TRIM_RULES:
        ps_v = ps_lr if ps_type == "lr" else T_hat
        mask = (ps_v >= lo) & (ps_v <= hi)
        n_keep = int(mask.sum())
        n_t    = int(T[mask].sum())
        n_c    = n_keep - n_t

        X_t = X[mask].reset_index(drop=True)
        Y_t = Y[mask]; T_t = T[mask]
        kf_t = make_kf()
        Y_hat_t, T_hat_t, Y_res_t, T_res_t = cross_fit(X_t, Y_t, T_t, kf=kf_t)
        plr_t  = plr_ate(Y_res_t, T_res_t)

        we_t  = we[mask]
        b4_t  = pd.cut(pd.Series(we_t), bins=EXP4_BINS, labels=EXP4_LABELS,
                       right=True, include_lowest=True)
        eff_t, joint_t, cont_t = band_effects_residual(Y_res_t, T_res_t, b4_t, EXP4_LABELS)

        # n per band per treatment
        band_ns = {lbl: (int((b4_t==lbl).sum()),
                         int(T_t[b4_t.values==lbl].sum()),
                         int((1-T_t)[b4_t.values==lbl].sum()))
                   for lbl in EXP4_LABELS}

        print(f"\n  {yr} · {rule_name}  n={n_keep:,} (T={n_t:,} C={n_c:,})"
              f"  ATE={plr_t['ate']:+.3f}  [{plr_t['ci_lo']:+.3f},{plr_t['ci_hi']:+.3f}]"
              f"  Wald χ²({joint_t['df']})={joint_t['chi2']:.3f}  p={joint_t['p']:.3f}")
        for lbl in EXP4_LABELS:
            nn, nt, nc = band_ns[lbl]
            print(f"    {lbl:12s}  n={nn:,} (T={nt:,} C={nc:,})"
                  f"  GATE={eff_t[lbl]['gate']:+.3f}  {_sig(eff_t[lbl]['p'])}")
        for name, c in cont_t.items():
            print(_fmt_contrast(f"  {name}", c))

        row = {"year": yr, "trim_rule": rule_name,
               "n_keep": n_keep, "n_treat": n_t, "n_ctrl": n_c,
               "ate": plr_t["ate"], "ate_ci_lo": plr_t["ci_lo"], "ate_ci_hi": plr_t["ci_hi"],
               "joint_chi2": joint_t["chi2"], "joint_p": joint_t["p"]}
        for lbl in EXP4_LABELS:
            row[f"gate_{lbl}"] = eff_t[lbl]["gate"]
            row[f"se_{lbl}"]   = eff_t[lbl]["se"]
        for name, c in cont_t.items():
            row[f"contrast_{name}_diff"] = c["diff"]
            row[f"contrast_{name}_p"]    = c["p"]
        rows_out.append(row)
        # 3-band for summary
        b3_t = pd.cut(pd.Series(we_t), bins=EXP3_BINS, labels=EXP3_LABELS,
                      right=True, include_lowest=True)
        eff3_t, joint3_t, cont3_t = band_effects_residual(Y_res_t, T_res_t, b3_t, EXP3_LABELS)
        band3_ns = {lbl: (int((b3_t == lbl).sum()),
                          int(T_t[b3_t.values == lbl].sum()),
                          int((1 - T_t)[b3_t.values == lbl].sum()))
                    for lbl in EXP3_LABELS}
        sec_c_summary[yr][rule_name] = dict(
            ate=plr_t["ate"], joint_p=joint_t["p"],
            n_keep=n_keep, n_treat=n_t, n_ctrl=n_c,
            eff3=eff3_t, joint3=joint3_t, cont3=cont3_t, band3_ns=band3_ns,
        )

    pd.DataFrame(rows_out).to_csv(REVDIR / f"trimming_{yr}.csv", index=False)

print("\nSection C done.")


Section C — Common support restrictions


/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),
/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),



  2025 · LR [0.05,0.95]  n=19,804 (T=15,425 C=4,379)  ATE=+0.121  [+0.053,+0.189]  Wald χ²(3)=6.035  p=0.110
    0-2 yrs       n=1,452 (T=1,219 C=233)  GATE=-0.177  ns
    3-5 yrs       n=3,038 (T=2,511 C=527)  GATE=+0.057  ns
    6-20 yrs      n=11,060 (T=8,695 C=2,365)  GATE=+0.182  ***
    21+ yrs       n=4,254 (T=3,000 C=1,254)  GATE=+0.087  ns
    6-20_vs_0-2         Δ=+0.359  [+0.041,+0.677]  p=0.027  *
    6-20_vs_21+         Δ=+0.095  [-0.062,+0.253]  p=0.236  ns


/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),
/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),



  2025 · LR [0.10,0.90]  n=17,246 (T=13,058 C=4,188)  ATE=+0.124  [+0.054,+0.194]  Wald χ²(3)=5.847  p=0.119
    0-2 yrs       n=1,159 (T=939 C=220)  GATE=-0.196  ns
    3-5 yrs       n=2,410 (T=1,922 C=488)  GATE=+0.085  ns
    6-20 yrs      n=9,579 (T=7,333 C=2,246)  GATE=+0.182  ***
    21+ yrs       n=4,098 (T=2,864 C=1,234)  GATE=+0.090  ns
    6-20_vs_0-2         Δ=+0.378  [+0.051,+0.704]  p=0.023  *
    6-20_vs_21+         Δ=+0.092  [-0.069,+0.252]  p=0.263  ns


/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),
/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),


... (3 outputs left out)

/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),
/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),



  2026 · LR [0.10,0.90]  n=5,679 (T=4,476 C=1,203)  ATE=+0.165  [+0.024,+0.306]  Wald χ²(3)=6.642  p=0.084
    0-2 yrs       n=492 (T=372 C=120)  GATE=-0.311  ns
    3-5 yrs       n=731 (T=587 C=144)  GATE=+0.086  ns
    6-20 yrs      n=2,781 (T=2,166 C=615)  GATE=+0.312  **
    21+ yrs       n=1,675 (T=1,351 C=324)  GATE=+0.097  ns
    6-20_vs_0-2         Δ=+0.623  [+0.124,+1.122]  p=0.014  *
    6-20_vs_21+         Δ=+0.215  [-0.122,+0.551]  p=0.211  ns



  2026 · LGBM [0.05,0.95]  n=5,629 (T=4,548 C=1,081)  ATE=+0.100  [-0.047,+0.247]  Wald χ²(3)=6.007  p=0.111
    0-2 yrs       n=459 (T=358 C=101)  GATE=-0.400  ns
    3-5 yrs       n=678 (T=550 C=128)  GATE=+0.156  ns
    6-20 yrs      n=2,859 (T=2,290 C=569)  GATE=+0.212  *
    21+ yrs       n=1,633 (T=1,350 C=283)  GATE=+0.033  ns
    6-20_vs_0-2         Δ=+0.612  [+0.111,+1.114]  p=0.017  *
    6-20_vs_21+         Δ=+0.178  [-0.181,+0.537]  p=0.330  ns

Section C done.


/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),
/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),


## Section D — Predetermined-Covariates Specification

In [7]:
print("=" * 70)
print("Section D — Predetermined-covariates spec vs full spec")
print("=" * 70)

sec_d_summary = {}

for yr in [2025, 2026]:
    path = YEARS[yr]
    d    = year_data[yr]
    df, X_full, Y, T = d["df"], d["X"], d["Y"], d["T"]
    kf_smpls = d["kf_smpls"]

    # Pre-spec: Age, EdLevel, WorkExp, YearsCode, Country, two missings
    PRE_NUM = ["Age", "EdLevel", "WorkExp", "YearsCode", "WorkExp_missing", "YearsCode_missing"]
    X_num_pre = df[PRE_NUM].astype(float)
    # Country already collapsed in df (by load_year)
    X_nom_pre = pd.get_dummies(df[["Country"]], drop_first=False, dtype=float)
    X_pre = pd.concat([X_num_pre, X_nom_pre], axis=1).reset_index(drop=True)

    we  = df["WorkExp"].values
    b4  = pd.cut(pd.Series(we), bins=EXP4_BINS, labels=EXP4_LABELS,
                 right=True, include_lowest=True)

    sec_d_summary[yr] = {}
    rows_out = []

    for spec_name, X_sp in [("full", X_full), ("pre", X_pre)]:
        kf_sp = make_kf()
        kf_smpls_sp = list(kf_sp.split(X_sp))
        Y_hat_sp, T_hat_sp, Y_res_sp, T_res_sp = cross_fit(X_sp, Y, T, kf=kf_sp)
        plr_sp  = plr_ate(Y_res_sp, T_res_sp)
        irm_sp  = run_irm(X_sp, Y, T, kf_smpls_sp, score="ATE", trim=0.01)
        eff_sp, joint_sp, cont_sp = band_effects_residual(
            Y_res_sp, T_res_sp, b4, EXP4_LABELS)

        print(f"\n  {yr} · {spec_name} spec  "
              f"PLR={plr_sp['ate']:+.3f}  IRM={irm_sp['ate']:+.3f}"
              f"  Wald χ²({joint_sp['df']})={joint_sp['chi2']:.3f}  p={joint_sp['p']:.3f}")
        for lbl in EXP4_LABELS:
            print(_fmt_band(lbl, eff_sp[lbl]))
        for name, c in cont_sp.items():
            print(_fmt_contrast(name, c))

        # 3-band for summary
        b3_sp = pd.cut(pd.Series(we), bins=EXP3_BINS, labels=EXP3_LABELS,
                       right=True, include_lowest=True)
        eff3_sp, joint3_sp, cont3_sp = band_effects_residual(
            Y_res_sp, T_res_sp, b3_sp, EXP3_LABELS)
        sec_d_summary[yr][spec_name] = dict(
            plr=plr_sp["ate"], irm=irm_sp["ate"],
            joint_p=joint_sp["p"],
            joint3_p=joint3_sp["p"],
            eff3=eff3_sp, cont3=cont3_sp,
        )
        row = {"year": yr, "spec": spec_name,
               "plr_ate": plr_sp["ate"], "plr_ci_lo": plr_sp["ci_lo"], "plr_ci_hi": plr_sp["ci_hi"],
               "irm_ate": irm_sp["ate"], "irm_ci_lo": irm_sp["ci_lo"], "irm_ci_hi": irm_sp["ci_hi"],
               "joint_chi2": joint_sp["chi2"], "joint_p": joint_sp["p"]}
        for lbl in EXP4_LABELS:
            row[f"gate_{lbl}"] = eff_sp[lbl]["gate"]
        for name, c in cont_sp.items():
            row[f"contrast_{name}_diff"] = c["diff"]
            row[f"contrast_{name}_p"]    = c["p"]
        rows_out.append(row)

    pd.DataFrame(rows_out).to_csv(REVDIR / f"prespec_{yr}.csv", index=False)

print("\nSection D done.")


Section D — Predetermined-covariates spec vs full spec


/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),
/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),



  2025 · full spec  PLR=+0.143  IRM=+0.221  Wald χ²(3)=5.547  p=0.136
  0-2 yrs       n= 1,539  GATE=-0.111  [-0.415,+0.193]  ns
  3-5 yrs       n= 3,217  GATE=+0.133  [-0.064,+0.330]  ns
  6-20 yrs      n=11,341  GATE=+0.204  [+0.112,+0.297]  ***
  21+ yrs       n= 4,276  GATE=+0.073  [-0.056,+0.201]  ns
  6-20_vs_0-2           Δ=+0.315  [-0.002,+0.633]  p=0.051  ns
  6-20_vs_21+           Δ=+0.132  [-0.027,+0.290]  p=0.103  ns


/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),
/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),



  2025 · pre spec  PLR=+0.169  IRM=+0.143  Wald χ²(3)=5.895  p=0.117
  0-2 yrs       n= 1,539  GATE=-0.130  [-0.428,+0.167]  ns
  3-5 yrs       n= 3,217  GATE=+0.166  [-0.028,+0.359]  ns
  6-20 yrs      n=11,341  GATE=+0.224  [+0.133,+0.315]  ***
  21+ yrs       n= 4,276  GATE=+0.120  [-0.007,+0.247]  ns
  6-20_vs_0-2           Δ=+0.355  [+0.044,+0.666]  p=0.025  *
  6-20_vs_21+           Δ=+0.104  [-0.052,+0.260]  p=0.191  ns


/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),
/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),



  2026 · full spec  PLR=+0.212  IRM=+0.551  Wald χ²(3)=9.749  p=0.021
  0-2 yrs       n=   795  GATE=-0.003  [-0.442,+0.435]  ns
  3-5 yrs       n= 1,441  GATE=+0.143  [-0.211,+0.497]  ns
  6-20 yrs      n= 6,132  GATE=+0.398  [+0.221,+0.575]  ***
  21+ yrs       n= 2,894  GATE=-0.029  [-0.252,+0.195]  ns
  6-20_vs_0-2           Δ=+0.401  [-0.072,+0.875]  p=0.096  ns
  6-20_vs_21+           Δ=+0.427  [+0.142,+0.712]  p=0.003  **



  2026 · pre spec  PLR=+0.170  IRM=+0.237  Wald χ²(3)=7.673  p=0.053
  0-2 yrs       n=   795  GATE=-0.032  [-0.437,+0.372]  ns
  3-5 yrs       n= 1,441  GATE=+0.019  [-0.305,+0.343]  ns
  6-20 yrs      n= 6,132  GATE=+0.340  [+0.162,+0.518]  ***
  21+ yrs       n= 2,894  GATE=-0.007  [-0.231,+0.217]  ns
  6-20_vs_0-2           Δ=+0.373  [-0.069,+0.815]  p=0.098  ns
  6-20_vs_21+           Δ=+0.348  [+0.061,+0.634]  p=0.017  *

Section D done.


/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),
/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),


## Section E — Replication Comparability

2025 main / 2025 narrow / 2026 side-by-side.

In [8]:
print("=" * 70)
print("Section E — Replication comparability table")
print("=" * 70)

cols = {}   # col_name -> dict of row -> value

def _add_col(name, df_col, X_col, Y_col, T_col, narrow_label=None):
    kf_e = make_kf()
    Yh, Th, Yr, Tr = cross_fit(X_col, Y_col, T_col, kf=kf_e)
    plr_e = plr_ate(Yr, Tr)
    kf_smpls_e = list(kf_e.split(X_col))
    irm_e = run_irm(X_col, Y_col, T_col, kf_smpls_e, score="ATE", trim=0.01)
    we_e  = df_col["WorkExp"].values
    b3_e  = pd.cut(pd.Series(we_e), bins=EXP3_BINS, labels=EXP3_LABELS,
                   right=True, include_lowest=True)
    eff_e, joint_e, cont_e = band_effects_residual(Yr, Tr, b3_e, EXP3_LABELS)
    ps_e  = lr_ps(X_col, T_col)
    ext_e = ((ps_e < 0.05) | (ps_e > 0.95)).mean()

    n_col = len(Y_col)
    n_c_col = int((T_col == 0).sum())
    row = {
        "n":            n_col,
        "n_control":    n_c_col,
        "plr_ate":      plr_e["ate"],
        "plr_ci":       f"[{plr_e['ci_lo']:+.3f},{plr_e['ci_hi']:+.3f}]",
        "irm_ate":      irm_e["ate"],
        "gate_0-2":     eff_e["0-2 yrs"]["gate"],
        "gate_3-20":    eff_e["3-20 yrs"]["gate"],
        "gate_21+":     eff_e["21+ yrs"]["gate"],
        "mid_vs_jun":   cont_e["mid_vs_junior"]["diff"],
        "mid_vs_sen":   cont_e["mid_vs_senior"]["diff"],
        "joint_p":      joint_e["p"],
        "extreme_ps":   ext_e,
    }
    cols[name] = row
    print(f"  {name}: PLR={plr_e['ate']:+.3f}  IRM={irm_e['ate']:+.3f}"
          f"  extreme_ps={ext_e:.1%}")

# 2025 main
d25 = year_data[2025]
_add_col("2025 main", d25["df"], d25["X"], d25["Y"], d25["T"])

# 2025 narrow control
df_n = d25["df_narrow"]
cv_n = df_n["Country"].value_counts()
df_n2 = df_n.copy()
df_n2["Country"] = df_n2["Country"].where(~df_n2["Country"].isin(cv_n[cv_n<50].index), "Other")
X_n2 = pd.concat([df_n2[ORDINAL+NUMERIC+MISS_IND].astype(float),
                   pd.get_dummies(df_n2[NOMINAL], drop_first=False, dtype=float)
                  ], axis=1).reset_index(drop=True)
Y_n2 = df_n2["JobSat"].values.astype(float)
T_n2 = df_n2["treated"].values.astype(float)
_add_col("2025 narrow", df_n2, X_n2, Y_n2, T_n2)

# 2026
d26 = year_data[2026]
_add_col("2026", d26["df"], d26["X"], d26["Y"], d26["T"])

# Print table
rows_out = list(cols["2025 main"].keys())
print(f"\n  {'Row':20s}  {'2025 main':>12s}  {'2025 narrow':>12s}  {'2026':>12s}")
print(f"  {'─'*20}  {'─'*12}  {'─'*12}  {'─'*12}")
for rk in rows_out:
    v1 = cols["2025 main"][rk]
    v2 = cols["2025 narrow"][rk]
    v3 = cols["2026"][rk]
    def _fv(v):
        if isinstance(v, float): return f"{v:+.3f}"
        elif isinstance(v, int): return f"{v:,}"
        return str(v)
    print(f"  {rk:20s}  {_fv(v1):>12s}  {_fv(v2):>12s}  {_fv(v3):>12s}")

# Save
df_e = pd.DataFrame(cols).T
df_e.to_csv(REVDIR / "replication_comparability.csv")
print("\nSaved → replication_comparability.csv")


Section E — Replication comparability table


/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),


  2025 main: PLR=+0.143  IRM=+0.221  extreme_ps=2.8%


/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),


  2025 narrow: PLR=+0.124  IRM=+0.172  extreme_ps=11.6%


/var/folders/tx/4lmfklxn1v1b390w_hlxcktc0000gn/T/ipykernel_59061/523742532.py:181: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return dict(diff=float(t.effect[0]), se=float(t.sd[0]),


  2026: PLR=+0.212  IRM=+0.551  extreme_ps=22.5%

  Row                      2025 main   2025 narrow          2026
  ────────────────────  ────────────  ────────────  ────────────
  n                           20,373        19,320        11,262
  n_control                    4,402         3,349         1,479
  plr_ate                     +0.143        +0.124        +0.212
  plr_ci                [+0.075,+0.212]  [+0.047,+0.200]  [+0.087,+0.336]
  irm_ate                     +0.221        +0.172        +0.551
  gate_0-2                    -0.111        -0.249        -0.003
  gate_3-20                   +0.191        +0.208        +0.351
  gate_21+                    +0.073        -0.013        -0.029
  mid_vs_jun                  +0.302        +0.457        +0.354
  mid_vs_sen                  +0.118        +0.221        +0.379
  joint_p                     +0.078        +0.002        +0.016
  extreme_ps                  +0.028        +0.116        +0.225

Saved → replication_comparabil

## Section F — Nuisance Fit Quality

In [9]:
print("=" * 70)
print("Section F — Nuisance fit (cross-fitted R² and AUC)")
print("=" * 70)

rows_out = []
for yr in [2025, 2026]:
    d = year_data[yr]
    Y, T, Y_hat, T_hat = d["Y"], d["T"], d["Y_hat"], d["T_hat"]
    Y_res, T_res = d["Y_res"], d["T_res"]

    r2_y = float(1 - np.var(Y_res) / np.var(Y))
    r2_t = float(1 - np.var(T_res) / np.var(T))
    auc  = float(roc_auc_score(T, T_hat))

    print(f"  {yr}: R²(Y)={r2_y:.3f}  R²(T)={r2_t:.3f}  AUC(T)={auc:.3f}")
    rows_out.append({"year": yr, "r2_outcome": r2_y, "r2_treatment": r2_t, "auc_treatment": auc})

pd.DataFrame(rows_out).to_csv(REVDIR / "nuisance_fit.csv", index=False)
print("Saved → nuisance_fit.csv")


Section F — Nuisance fit (cross-fitted R² and AUC)
  2025: R²(Y)=0.001  R²(T)=0.052  AUC(T)=0.674
  2026: R²(Y)=-0.004  R²(T)=0.021  AUC(T)=0.686
Saved → nuisance_fit.csv


## Summary (copy-paste block)

In [10]:
print("=" * 72)
print("SUMMARY  (3 decimals; p to 3 decimals)")
print("=" * 72)

for yr in [2025, 2026]:
    d      = year_data[yr]
    rows_a = sec_a_rows[yr]
    sb     = sec_b_summary[yr]

    treat_share = float(d["T"].mean())

    j3r = sb["3-band"]["residual"]["joint"]
    j4r = sb["4-band"]["residual"]["joint"]
    j3d = sb["3-band"]["dr"]["joint"]
    j4d = sb["4-band"]["dr"]["joint"]
    c3r = sb["3-band"]["residual"]["contrasts"]
    c4r = sb["4-band"]["residual"]["contrasts"]
    e3r = sb["3-band"]["residual"]["effects"]
    e4r = sb["4-band"]["residual"]["effects"]
    e3d = sb["3-band"]["dr"]["effects"]
    e4d = sb["4-band"]["dr"]["effects"]
    c3d = sb["3-band"]["dr"]["contrasts"]
    c4d = sb["4-band"]["dr"]["contrasts"]

    print(f"\n{'─'*72}")
    print(f"  {yr}   treated share = {treat_share:.3f}")
    print(f"{'─'*72}")

    # ── Section A ─────────────────────────────────────────────────────────
    print(f"\n  SECTION A — AIPW/IRM decomposition")
    print(f"  {'Estimator':50s}  {'ATE':>7s}  {'SE':>6s}  {'95% CI':>18s}")
    print(f"  {'─'*50}  {'─'*7}  {'─'*6}  {'─'*18}")
    for r in rows_a:
        ci = f"[{r['ci_lo']:+.3f},{r['ci_hi']:+.3f}]"
        print(f"  {r['label']:50s}  {r['ate']:+.3f}  {r['se']:.3f}  {ci}")

    # ── Section B residual ────────────────────────────────────────────────
    print(f"\n  SECTION B — 3-band residual  Wald χ²({j3r['df']})={j3r['chi2']:.3f}  p={j3r['p']:.3f}  {_sig(j3r['p'])}")
    for lbl in EXP3_LABELS:
        e = e3r[lbl]
        print(f"    {lbl:12s}  GATE={e['gate']:+.3f}  [{e['ci_lo']:+.3f},{e['ci_hi']:+.3f}]  {_sig(e['p'])}")
    print(f"    mid vs junior  Δ={c3r['mid_vs_junior']['diff']:+.3f}  [{c3r['mid_vs_junior']['ci_lo']:+.3f},{c3r['mid_vs_junior']['ci_hi']:+.3f}]  p={c3r['mid_vs_junior']['p']:.3f}  {_sig(c3r['mid_vs_junior']['p'])}")
    print(f"    mid vs senior  Δ={c3r['mid_vs_senior']['diff']:+.3f}  [{c3r['mid_vs_senior']['ci_lo']:+.3f},{c3r['mid_vs_senior']['ci_hi']:+.3f}]  p={c3r['mid_vs_senior']['p']:.3f}  {_sig(c3r['mid_vs_senior']['p'])}")

    print(f"  SECTION B — 4-band residual  Wald χ²({j4r['df']})={j4r['chi2']:.3f}  p={j4r['p']:.3f}  {_sig(j4r['p'])}")
    for lbl in EXP4_LABELS:
        e = e4r[lbl]
        print(f"    {lbl:12s}  GATE={e['gate']:+.3f}  [{e['ci_lo']:+.3f},{e['ci_hi']:+.3f}]  {_sig(e['p'])}")
    print(f"    6-20 vs 0-2  Δ={c4r['6-20_vs_0-2']['diff']:+.3f}  [{c4r['6-20_vs_0-2']['ci_lo']:+.3f},{c4r['6-20_vs_0-2']['ci_hi']:+.3f}]  p={c4r['6-20_vs_0-2']['p']:.3f}  {_sig(c4r['6-20_vs_0-2']['p'])}")
    print(f"    6-20 vs 21+  Δ={c4r['6-20_vs_21+']['diff']:+.3f}  [{c4r['6-20_vs_21+']['ci_lo']:+.3f},{c4r['6-20_vs_21+']['ci_hi']:+.3f}]  p={c4r['6-20_vs_21+']['p']:.3f}  {_sig(c4r['6-20_vs_21+']['p'])}")

    # ── Section B DR-score ────────────────────────────────────────────────
    print(f"  SECTION B — 3-band DR-score  Wald χ²({j3d['df']})={j3d['chi2']:.3f}  p={j3d['p']:.3f}  {_sig(j3d['p'])}")
    for lbl in EXP3_LABELS:
        e = e3d[lbl]
        print(f"    {lbl:12s}  GATE={e['gate']:+.3f}  [{e['ci_lo']:+.3f},{e['ci_hi']:+.3f}]  {_sig(e['p'])}")
    print(f"    mid vs junior  Δ={c3d['mid_vs_junior']['diff']:+.3f}  [{c3d['mid_vs_junior']['ci_lo']:+.3f},{c3d['mid_vs_junior']['ci_hi']:+.3f}]  p={c3d['mid_vs_junior']['p']:.3f}  {_sig(c3d['mid_vs_junior']['p'])}")
    print(f"    mid vs senior  Δ={c3d['mid_vs_senior']['diff']:+.3f}  [{c3d['mid_vs_senior']['ci_lo']:+.3f},{c3d['mid_vs_senior']['ci_hi']:+.3f}]  p={c3d['mid_vs_senior']['p']:.3f}  {_sig(c3d['mid_vs_senior']['p'])}")

    print(f"  SECTION B — 4-band DR-score  Wald χ²({j4d['df']})={j4d['chi2']:.3f}  p={j4d['p']:.3f}  {_sig(j4d['p'])}")
    for lbl in EXP4_LABELS:
        e = e4d[lbl]
        print(f"    {lbl:12s}  GATE={e['gate']:+.3f}  [{e['ci_lo']:+.3f},{e['ci_hi']:+.3f}]  {_sig(e['p'])}")
    print(f"    6-20 vs 0-2  Δ={c4d['6-20_vs_0-2']['diff']:+.3f}  [{c4d['6-20_vs_0-2']['ci_lo']:+.3f},{c4d['6-20_vs_0-2']['ci_hi']:+.3f}]  p={c4d['6-20_vs_0-2']['p']:.3f}  {_sig(c4d['6-20_vs_0-2']['p'])}")
    print(f"    6-20 vs 21+  Δ={c4d['6-20_vs_21+']['diff']:+.3f}  [{c4d['6-20_vs_21+']['ci_lo']:+.3f},{c4d['6-20_vs_21+']['ci_hi']:+.3f}]  p={c4d['6-20_vs_21+']['p']:.3f}  {_sig(c4d['6-20_vs_21+']['p'])}")

    # ── Section C ─────────────────────────────────────────────────────────
    print(f"\n  SECTION C — Common support restrictions (3-band)")
    for rule, v in sec_c_summary[yr].items():
        j3 = v['joint3']
        print(f"    {rule:22s}  n={v['n_keep']:,} (T={v['n_treat']:,} C={v['n_ctrl']:,})"
              f"  ATE={v['ate']:+.3f}  Wald χ²({j3['df']})={j3['chi2']:.3f}  p={j3['p']:.3f}  {_sig(j3['p'])}")
        for lbl in EXP3_LABELS:
            nn, nt, nc = v['band3_ns'][lbl]
            e = v['eff3'][lbl]
            print(f"      {lbl:12s}  n={nn:,} (T={nt:,} C={nc:,})"
                  f"  GATE={e['gate']:+.3f}  [{e['ci_lo']:+.3f},{e['ci_hi']:+.3f}]  {_sig(e['p'])}")
        c = v['cont3']
        print(f"      mid vs junior  Δ={c['mid_vs_junior']['diff']:+.3f}  [{c['mid_vs_junior']['ci_lo']:+.3f},{c['mid_vs_junior']['ci_hi']:+.3f}]  p={c['mid_vs_junior']['p']:.3f}  {_sig(c['mid_vs_junior']['p'])}")
        print(f"      mid vs senior  Δ={c['mid_vs_senior']['diff']:+.3f}  [{c['mid_vs_senior']['ci_lo']:+.3f},{c['mid_vs_senior']['ci_hi']:+.3f}]  p={c['mid_vs_senior']['p']:.3f}  {_sig(c['mid_vs_senior']['p'])}")

    # ── Section D ─────────────────────────────────────────────────────────
    print(f"\n  SECTION D — Predetermined vs full spec (3-band)")
    for spec, v in sec_d_summary[yr].items():
        j4p = v['joint_p']; j3p = v['joint3_p']
        print(f"    {spec:8s}  PLR={v['plr']:+.3f}  IRM={v['irm']:+.3f}"
              f"  4-band joint p={j4p:.3f}  3-band joint p={j3p:.3f}  {_sig(j3p)}")
        for lbl in EXP3_LABELS:
            e = v['eff3'][lbl]
            print(f"      {lbl:12s}  GATE={e['gate']:+.3f}  [{e['ci_lo']:+.3f},{e['ci_hi']:+.3f}]  {_sig(e['p'])}")
        c = v['cont3']
        print(f"      mid vs junior  Δ={c['mid_vs_junior']['diff']:+.3f}  [{c['mid_vs_junior']['ci_lo']:+.3f},{c['mid_vs_junior']['ci_hi']:+.3f}]  p={c['mid_vs_junior']['p']:.3f}  {_sig(c['mid_vs_junior']['p'])}")
        print(f"      mid vs senior  Δ={c['mid_vs_senior']['diff']:+.3f}  [{c['mid_vs_senior']['ci_lo']:+.3f},{c['mid_vs_senior']['ci_hi']:+.3f}]  p={c['mid_vs_senior']['p']:.3f}  {_sig(c['mid_vs_senior']['p'])}")

    # ── Section F ─────────────────────────────────────────────────────────
    dd = year_data[yr]
    r2y = float(1 - np.var(dd["Y_res"]) / np.var(dd["Y"]))
    r2t = float(1 - np.var(dd["T_res"]) / np.var(dd["T"]))
    auc = float(roc_auc_score(dd["T"], dd["T_hat"]))
    print(f"\n  SECTION F  R²(Y)={r2y:.3f}  R²(T)={r2t:.3f}  AUC(T)={auc:.3f}")

# ── Section E — full comparability table ─────────────────────────────────────
print(f"\n{'─'*72}")
print("  SECTION E — Replication comparability (full table)")
print(f"{'─'*72}")
df_e = pd.read_csv(REVDIR / "replication_comparability.csv")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.3f}".format)
print(df_e.to_string(index=False))

print("\n" + "="*72)


SUMMARY  (3 decimals; p to 3 decimals)

────────────────────────────────────────────────────────────────────────
  2025   treated share = 0.784
────────────────────────────────────────────────────────────────────────

  SECTION A — AIPW/IRM decomposition
  Estimator                                               ATE      SE              95% CI
  ──────────────────────────────────────────────────  ───────  ──────  ──────────────────
  1. PLR (main)                                       +0.143  0.035  [+0.075,+0.212]
  2. AIPW S-learner clip=0.05 wins=yes                +0.011  0.028  [-0.044,+0.067]
  3. AIPW S-learner clip=0.05 wins=no                 +0.163  0.053  [+0.059,+0.266]
  4. AIPW S-learner clip=0.01 wins=yes                +0.015  0.029  [-0.041,+0.072]
  5. AIPW S-learner clip=0.01 wins=no                 +0.227  0.078  [+0.073,+0.380]
  6. AIPW T-learner clip=0.05 wins=yes                +0.034  0.030  [-0.024,+0.092]
  7. AIPW T-learner clip=0.05 wins=no                 +